# NB-01-1 パウリ行列の性質を、行列計算で確かめる

[パウリ行列のノート](../../研究ノート/04_群論・代数/pauli_matrices_derivation_and_group.md)の Part II・IV・V で手で計算した性質を、NumPy の行列計算で確かめます。最後に、同じことを Qiskit の `quantum_info` でも確かめて、答え合わせをします。

**確かめること**

1. パウリ行列は、トレースゼロのエルミート行列の一般形から取り出せる（ノートの Part II §3）
2. 基本性質の表：エルミート、$\sigma^2=I$、ユニタリ、トレース $0$、行列式 $-1$、固有値 $\pm1$（Part II §6）
3. 積の表 $\sigma_i\sigma_j=\delta_{ij}I+i\varepsilon_{ijk}\sigma_k$、交換関係、反交換関係（Part IV・V）
4. ベクトルとの積 $(\vec a\cdot\vec\sigma)(\vec b\cdot\vec\sigma)=(\vec a\cdot\vec b)I+i(\vec a\times\vec b)\cdot\vec\sigma$（Part V §3。スライダーで $\vec b$ を動かせます）
5. 任意の $2\times2$ 行列の展開 $M=\frac12\operatorname{tr}(M)I+\sum_i\frac12\operatorname{tr}(\sigma_iM)\sigma_i$（Part II §5）

**使い方**：上から順に実行します。各セルの `assert` が、ノートの式と計算結果の一致を確かめます（一致しなければエラーで止まります）。

In [ ]:
import numpy as np
import plotly.graph_objects as go
import ipywidgets as W
from numpy.typing import NDArray
from plotly.subplots import make_subplots

np.set_printoptions(precision=4, suppress=True)

# 型の別名（型ヒントを読みやすくするため。実行時には何も確かめない）
Matrix = NDArray[np.complex128]   # 2×2 の複素行列
Vector = NDArray[np.float64]      # 3次元の実ベクトル


def traceless_hermitian(nx: float, ny: float, nz: float) -> Matrix:
    """ノートの Part II §3：トレースゼロのエルミート行列の一般形"""
    return np.array([[nz, nx - 1j * ny], [nx + 1j * ny, -nz]])


# 係数を1つずつ 1 にすると、パウリ行列が取り出せる
sx = traceless_hermitian(1, 0, 0)
sy = traceless_hermitian(0, 1, 0)
sz = traceless_hermitian(0, 0, 1)
I2: Matrix = np.eye(2, dtype=complex)
NAMES = "xyz"
S: list[Matrix] = [sx, sy, sz]

for name, M in zip(NAMES, S):
    print(f"σ{name} =\n{M}\n")

## 1. 基本性質の表

ノートの Part II §6 の表を、1行ずつ確かめます。固有ベクトルは、表に書いた形（$\sigma_x$ なら $\frac1{\sqrt2}(1,\pm1)$ など）で、本当に固有値 $\pm1$ の固有ベクトルになっているかを、$\sigma\mathbf v=\pm\mathbf v$ で確かめます。

In [ ]:
EIGVEC: dict[str, tuple[list[complex], list[complex]]] = {"x": ([1, 1], [1, -1]), "y": ([1, 1j], [1, -1j]), "z": ([1, 0], [0, 1])}

for name, M in zip(NAMES, S):
    herm = np.allclose(M, M.conj().T)
    square = np.allclose(M @ M, I2)
    unitary = np.allclose(M.conj().T @ M, I2)
    tr, det = np.trace(M).real, np.linalg.det(M).real
    eig = np.linalg.eigvalsh(M)
    vp, vm = (np.array(v, dtype=complex) / np.linalg.norm(v) for v in EIGVEC[name])
    print(f"σ{name}: エルミート {herm}、σ²=I {square}、ユニタリ {unitary}、tr = {tr:+.0f}、det = {det:+.0f}、固有値 {eig}")
    assert herm and square and unitary and np.isclose(tr, 0) and np.isclose(det, -1)
    assert np.allclose(eig, [-1, 1])
    assert np.allclose(M @ vp, vp) and np.allclose(M @ vm, -vm)

# iσ は SU(2) に入る（det = 1）が、σ 自身は入らない（det = −1）
assert all(np.isclose(np.linalg.det(1j * M), 1) for M in S)
print("\nすべての性質を確認しました")

## 2. 積の表と、交換関係・反交換関係

レヴィ＝チヴィタ記号 $\varepsilon_{ijk}$ を、「$(i,j,k)$ を並べ替える置換の符号」として計算し、9通りの積がすべて $\sigma_i\sigma_j=\delta_{ij}I+i\varepsilon_{ijk}\sigma_k$ になることを確かめます（ノートの Part IV §5 の表）。

In [ ]:
def eps(i: int, j: int, k: int) -> int:
    """レヴィ＝チヴィタ記号（x, y, z を 0, 1, 2 とする）。置換行列の行列式が置換の符号"""
    if len({i, j, k}) < 3:
        return 0
    return int(round(np.linalg.det(np.eye(3)[[i, j, k]])))


def label(M: Matrix) -> str:
    """行列を I, ±iσk のどれかの名前で表す（積の表の表示用）"""
    if np.allclose(M, I2):
        return "I".center(6)
    for k in range(3):
        for c, s in [(1j, "+iσ"), (-1j, "-iσ")]:
            if np.allclose(M, c * S[k]):
                return f"{s}{NAMES[k]}".center(6)
    return "?".center(6)


print("σiσj |" + "".join(f"σ{n}".center(6) for n in NAMES))
for i in range(3):
    row = []
    for j in range(3):
        P = S[i] @ S[j]
        formula = (i == j) * I2 + 1j * sum(eps(i, j, k) * S[k] for k in range(3))
        assert np.allclose(P, formula)
        row.append(label(P))
    print(f" σ{NAMES[i]}  |" + "".join(row))

for i in range(3):
    for j in range(3):
        comm = S[i] @ S[j] - S[j] @ S[i]
        anti = S[i] @ S[j] + S[j] @ S[i]
        assert np.allclose(comm, 2j * sum(eps(i, j, k) * S[k] for k in range(3)))   # [σi, σj] = 2i ε σk
        assert np.allclose(anti, 2 * (i == j) * I2)                                # {σi, σj} = 2δ I
print("\n積の表、交換関係、反交換関係を確認しました")

## 3. ベクトルとの積：内積と外積

$\vec a\cdot\vec\sigma=a_x\sigma_x+a_y\sigma_y+a_z\sigma_z$ と書きます。まず、ランダムなベクトルで公式と系（2乗・交換子・反交換子）を確かめます。

In [ ]:
def dot_sigma(v: Vector) -> Matrix:
    return v[0] * sx + v[1] * sy + v[2] * sz


def coefficients(M: Matrix) -> NDArray[np.complex128]:
    """2×2 行列を I, σx, σy, σz で展開した係数（ノートの Part II §5：½tr で取り出す）"""
    return np.array([np.trace(M) / 2] + [np.trace(Sk @ M) / 2 for Sk in S])


rng = np.random.default_rng(1)
for _ in range(100):
    a, b = rng.normal(size=3), rng.normal(size=3)
    A, B = dot_sigma(a), dot_sigma(b)
    assert np.allclose(A @ B, np.dot(a, b) * I2 + 1j * dot_sigma(np.cross(a, b)))   # 公式
    assert np.allclose(A @ A, np.dot(a, a) * I2)                                   # (a·σ)² = |a|² I
    assert np.allclose(A @ B - B @ A, 2j * dot_sigma(np.cross(a, b)))               # 交換子は外積
    assert np.allclose(A @ B + B @ A, 2 * np.dot(a, b) * I2)                        # 反交換子は内積
c = coefficients(A @ B)
print("最後の例：I の係数", np.round(c[0], 4), " a·b =", round(np.dot(a, b), 4))
print("          σ の係数の虚部", np.round(c[1:].imag, 4), " a×b =", np.round(np.cross(a, b), 4), "（実部は", np.round(c[1:].real, 12), "）")

### スライダーで $\vec b$ を動かす

$\vec a=\hat x$（青）を固定し、$\vec b$（橙）の向きをスライダーで変えます。左の図に $\vec a\times\vec b$（赤）、右の図に積 $(\vec a\cdot\vec\sigma)(\vec b\cdot\vec\sigma)$ を行列として計算し、$\frac12\operatorname{tr}$ で取り出した係数を描きます。黒い横線は、公式から計算した $\vec a\cdot\vec b$ と $\vec a\times\vec b$ の成分です。

In [ ]:
a: Vector = np.array([1.0, 0.0, 0.0])


def vec_b(theta_deg: float, phi_deg: float) -> Vector:
    t, p = np.radians(theta_deg), np.radians(phi_deg)
    return np.array([np.sin(t) * np.cos(p), np.sin(t) * np.sin(p), np.cos(t)])


def arrow3d(v: Vector, color: str, name: str) -> go.Scatter3d:
    return go.Scatter3d(x=[0, v[0]], y=[0, v[1]], z=[0, v[2]], mode="lines+markers", name=name,
                        line=dict(color=color, width=7), marker=dict(size=[0, 5], color=color))


fig = go.FigureWidget(make_subplots(rows=1, cols=2, specs=[[{"type": "scene"}, {"type": "xy"}]],
                                    column_widths=[0.55, 0.45], subplot_titles=("ベクトル", "積の係数（½tr で取り出す）")))
fig.add_trace(arrow3d(a, "#1f77b4", "a"), 1, 1)
fig.add_trace(arrow3d(a, "#e67e00", "b"), 1, 1)
fig.add_trace(arrow3d(a, "#d62728", "a×b"), 1, 1)
fig.add_trace(go.Bar(x=["I（実部）", "σx（虚部）", "σy（虚部）", "σz（虚部）"], y=[0, 0, 0, 0],
                     marker_color=["#1f77b4", "#d62728", "#d62728", "#d62728"], name="係数"), 1, 2)
fig.add_trace(go.Scatter(x=["I（実部）", "σx（虚部）", "σy（虚部）", "σz（虚部）"], y=[0, 0, 0, 0], mode="markers",
                         marker=dict(symbol="line-ew", size=40, line=dict(width=3, color="black")), name="公式"), 1, 2)
fig.update_layout(height=480, width=900, margin=dict(l=10, r=10, t=40, b=10),
                  scene=dict(xaxis=dict(range=[-1, 1]), yaxis=dict(range=[-1, 1]), zaxis=dict(range=[-1, 1]), aspectmode="cube",
                             camera=dict(eye=dict(x=1.5, y=-1.4, z=0.8))),
                  yaxis=dict(range=[-1.1, 1.1]))

theta_b = W.FloatSlider(value=60, min=0, max=180, step=5, description="b の θ")
phi_b = W.FloatSlider(value=40, min=-180, max=180, step=5, description="b の φ")
info = W.HTML()


def update(*_: object) -> None:
    """スライダーが動くたびに呼ばれ、図と文字の表示を描き直す"""
    b = vec_b(theta_b.value, phi_b.value)
    cr = np.cross(a, b)
    c = coefficients(dot_sigma(a) @ dot_sigma(b))
    assert np.isclose(c[0], np.dot(a, b)) and np.allclose(c[1:], 1j * cr)
    with fig.batch_update():
        for k, v in [(1, b), (2, cr)]:
            fig.data[k].x, fig.data[k].y, fig.data[k].z = [0, v[0]], [0, v[1]], [0, v[2]]
        fig.data[3].y = [c[0].real, *c[1:].imag]
        fig.data[4].y = [np.dot(a, b), *cr]
    info.value = (f"a·b = {np.dot(a, b):+.3f}、a×b = ({cr[0]:+.3f}, {cr[1]:+.3f}, {cr[2]:+.3f})　"
                  f"→ b が a に平行なら外積が 0、垂直なら内積が 0")


theta_b.observe(update, "value")
phi_b.observe(update, "value")
update()
W.VBox([W.HBox([theta_b, phi_b]), info, fig])

## 4. 任意の $2\times2$ 行列の展開と、Qiskit での答え合わせ

ランダムな複素行列 $M$ を、$\frac12\operatorname{tr}$ で $\{I,\sigma_x,\sigma_y,\sigma_z\}$ に展開して元に戻ることを確かめ、Qiskit の `SparsePauliOp.from_operator` が同じ係数を返すことを確かめます。ノートの Part II §4 の例 $M=\begin{pmatrix}1&2-i\\2+i&-1\end{pmatrix}=2\sigma_x+\sigma_y+\sigma_z$ も確かめます。

In [ ]:
from qiskit.quantum_info import Operator, Pauli, SparsePauliOp

examples = {"ランダムな複素行列": rng.normal(size=(2, 2)) + 1j * rng.normal(size=(2, 2)),
            "ノートの例": np.array([[1, 2 - 1j], [2 + 1j, -1]])}
for title, M in examples.items():
    c = coefficients(M)
    assert np.allclose(c[0] * I2 + sum(ck * Sk for ck, Sk in zip(c[1:], S)), M)     # 元に戻る
    op = SparsePauliOp.from_operator(Operator(M))
    qiskit_coef = dict(zip(op.paulis.to_labels(), op.coeffs))
    for k, label_ in enumerate("IXYZ"):
        assert np.isclose(qiskit_coef.get(label_, 0), c[k])
    print(f"{title}：自前の係数 {np.round(c, 4)}")
    print(f"{' ' * len(title)}  Qiskit     {op}\n")

## 5. Qiskit の `Pauli` で答え合わせ（積の順序に注意）

Qiskit の `Pauli` オブジェクトは、位相（$\pm1,\pm i$）まで込めてパウリ行列を扱います。行列が自前の定義と一致すること、積と交換関係が同じになることを確かめます。

**注意**：`P.dot(Q)`（と `P @ Q`）は行列の積 $PQ$ ですが、`P.compose(Q)` は「$P$ を作用させてから $Q$ を作用させる」という意味で、行列としては $QP$ です。

In [ ]:
PX, PY, PZ = Pauli("X"), Pauli("Y"), Pauli("Z")
for P, M in zip((PX, PY, PZ), S):
    assert np.allclose(P.to_matrix(), M)

print("X.dot(Y)     =", PX.dot(PY), "  （行列の積 σxσy = iσz）")
print("X @ Y        =", PX @ PY)
print("X.compose(Y) =", PX.compose(PY), "  （作用の順：X の次に Y、つまり σyσx = −iσz）")
assert np.allclose(PX.dot(PY).to_matrix(), sx @ sy)
assert np.allclose(PX.compose(PY).to_matrix(), sy @ sx)
assert PX.dot(PY) == Pauli("iZ")

print("\nX と Y は交換する？", PX.commutes(PY), "／反交換する？", PX.anticommutes(PY))
assert PX.anticommutes(PY) and PX.commutes(PX)

## 観察のポイント

- パウリ行列は「発明された行列」ではなく、トレースゼロのエルミート行列の一般形から、係数を1つずつ取り出しただけのものです（最初のセル）。
- 積の表は、$x\to y\to z\to x$ の向きに掛けると $+i$、逆向きなら $-i$ で、その符号が $\varepsilon_{ijk}$ です。
- スライダーで $\vec b$ を $\vec a$ に平行にすると、積は $(\vec a\cdot\vec b)I$ だけになり（外積が $0$）、垂直にすると $i(\vec a\times\vec b)\cdot\vec\sigma$ だけになります（内積が $0$）。
- Qiskit の `compose` は作用の順で、行列の積とは逆順です。Phase 2 で回路を組むときに、つまずきやすいところです。

**次に試すこと**：`traceless_hermitian` の代わりに、右上の成分の虚部を $n_y$ とする定義（$\sigma_y$ の符号が逆）にすると、どのセルの `assert` が止まるでしょうか（ノートの Part II §3 の「$\sigma_y$ の符号について」）。

**次のノートブック**：[02_bloch_sphere.ipynb](02_bloch_sphere.ipynb) で、$\hat n\cdot\vec\sigma$ の固有ベクトルとブロッホ球を扱います。